# ATLAS v2.0 → Kaggle Dataset Ingestion
**Run once.** Downloads raw ATLAS data from NITRC, extracts it, validates structure,
and organises it into a clean directory ready to be committed as a Kaggle Dataset.

**Before running:**
- Paste your NITRC download URL in the `Config` cell below.
- Enable Internet in Kaggle: *Settings → Internet → On*.
- This notebook writes to `/kaggle/working/atlas_raw/` (~4 GB).

## 0. Imports

In [1]:
import os
import hashlib
import tarfile
import zipfile
import shutil
import json
import subprocess
from pathlib import Path
import pandas as pd
from urllib.request import urlretrieve
from collections import defaultdict

print(f"Working dir: {Path.cwd()}")

Working dir: /kaggle/working


## 1. Config — edit this cell only

In [2]:
# ── EDIT BELOW ────────────────────────────────────────────────────────────────
NITRC_URL = "https://fcp-indi.s3.us-east-1.amazonaws.com/data/Projects/INDI/ATLAS/R2.1/atlas21_training_raw.tar.gz"   # full URL including any token
ARCHIVE_FILENAME = "atlas21_training_raw.tar.gz"         # The raw encrypted file name
DECRYPTION_KEY = "bLw,A>?jJ6j6KnV"                   # Put your OpenSSL password here
# ── END EDIT ──────────────────────────────────────────────────────────────────

WORK_DIR     = Path("/kaggle/working")
ARCHIVE_PATH = WORK_DIR / ARCHIVE_FILENAME
DECRYPTED_FILENAME = "ATLAS_R2.1_raw.tar.gz"             # The output clean filename after decryption
DECRYPTED_PATH = WORK_DIR / DECRYPTED_FILENAME
EXTRACT_DIR  = WORK_DIR / "atlas_extracted"
OUTPUT_DIR   = WORK_DIR / "atlas_raw"                    # final clean structure

for d in [EXTRACT_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

assert NITRC_URL != "PASTE_YOUR_NITRC_DOWNLOAD_URL_HERE", \
    "Set NITRC_URL before running."

print("Config OK")
print(f"  Archive   : {ARCHIVE_PATH}")
print(f"  Extract to: {EXTRACT_DIR}")
print(f"  Output    : {OUTPUT_DIR}")

Config OK
  Archive   : /kaggle/working/atlas21_training_raw.tar.gz
  Extract to: /kaggle/working/atlas_extracted
  Output    : /kaggle/working/atlas_raw


## 2. Download

In [3]:
def _progress(block_num, block_size, total_size):
    """Minimal download progress indicator."""
    downloaded = block_num * block_size
    if total_size > 0:
        pct = min(downloaded / total_size * 100, 100)
        if block_num % 10000 == 0:
            print(f"  {pct:.1f}%  ({downloaded / 1e6:.0f} MB / {total_size / 1e6:.0f} MB)",
                  flush=True)

if ARCHIVE_PATH.exists():
    print(f"Archive already present ({ARCHIVE_PATH.stat().st_size / 1e6:.0f} MB), skipping download.")
else:
    print(f"Downloading {ARCHIVE_FILENAME} ...")
    urlretrieve(NITRC_URL, ARCHIVE_PATH, reporthook=_progress)
    print(f"Download complete: {ARCHIVE_PATH.stat().st_size / 1e6:.0f} MB")

  0.0%  (0 MB / 3970 MB)
  2.1%  (82 MB / 3970 MB)
  4.1%  (164 MB / 3970 MB)
  6.2%  (246 MB / 3970 MB)
  8.3%  (328 MB / 3970 MB)
  10.3%  (410 MB / 3970 MB)
  12.4%  (492 MB / 3970 MB)
  14.4%  (573 MB / 3970 MB)
  16.5%  (655 MB / 3970 MB)
  18.6%  (737 MB / 3970 MB)
  20.6%  (819 MB / 3970 MB)
  22.7%  (901 MB / 3970 MB)
  24.8%  (983 MB / 3970 MB)
  26.8%  (1065 MB / 3970 MB)
  28.9%  (1147 MB / 3970 MB)
  31.0%  (1229 MB / 3970 MB)
  33.0%  (1311 MB / 3970 MB)
  35.1%  (1393 MB / 3970 MB)
  37.1%  (1475 MB / 3970 MB)
  39.2%  (1556 MB / 3970 MB)
  41.3%  (1638 MB / 3970 MB)
  43.3%  (1720 MB / 3970 MB)
  45.4%  (1802 MB / 3970 MB)
  47.5%  (1884 MB / 3970 MB)
  49.5%  (1966 MB / 3970 MB)
  51.6%  (2048 MB / 3970 MB)
  53.6%  (2130 MB / 3970 MB)
  55.7%  (2212 MB / 3970 MB)
  57.8%  (2294 MB / 3970 MB)
  59.8%  (2376 MB / 3970 MB)
  61.9%  (2458 MB / 3970 MB)
  64.0%  (2540 MB / 3970 MB)
  66.0%  (2621 MB / 3970 MB)
  68.1%  (2703 MB / 3970 MB)
  70.2%  (2785 MB / 3970 MB)
  72.2

## 3. Decrypt and Extract

In [4]:
def decrypt_file(encrypted_path: Path, decrypted_path: Path, key: str) -> None:
    """Decrypts OpenSSL AES-256-CBC base64 file using the given key."""
    if decrypted_path.exists():
        print("Decrypted file already exists. Skipping decryption step.")
        return

    print(f"Decrypting {encrypted_path.name} via OpenSSL...")
    # OpenSSL command provided by ISLES 2026. -k specifies the password programmatically.
    cmd = [
        "openssl", "aes-256-cbc", "-md", "sha256", "-d", "-a",
        "-in", str(encrypted_path),
        "-out", str(decrypted_path),
        "-k", key
    ]
    
    # Run the system shell execution cleanly
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        if decrypted_path.exists():
            os.remove(decrypted_path) # Clean up partial failures
        raise RuntimeError(f"OpenSSL Decryption failed! Check your password. Error:\n{result.stderr}")
    print("Decryption complete.")

def extract_archive(archive_path: Path, dest: Path) -> None:
    """Extract .zip or .tar.gz archive to dest."""
    filename = archive_path.name.lower()
    print(f"Extracting {archive_path.name} → {dest} ...")
    
    if filename.endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(dest)
    elif filename.endswith((".tar.gz", ".tgz")):
        with tarfile.open(archive_path, "r:gz") as tf:
            tf.extractall(dest)
    elif filename.endswith(".tar"):
        with tarfile.open(archive_path, "r:") as tf:
            tf.extractall(dest)
    else:
        # Fallback to get any actual extension left over for logging
        suffix = "".join(archive_path.suffixes)
        raise ValueError(f"Unsupported archive format: {suffix}")
        
    print("Extraction complete.")

# 1. Run Decryption first using OpenSSL
decrypt_file(ARCHIVE_PATH, DECRYPTED_PATH, DECRYPTION_KEY)

# 2. Extract the now-clean, decrypted tarball
if any(EXTRACT_DIR.iterdir()) if EXTRACT_DIR.exists() else False:
    print("Extract dir already populated, skipping extraction.")
else:
    extract_archive(DECRYPTED_PATH, EXTRACT_DIR)

Decrypting atlas21_training_raw.tar.gz via OpenSSL...
Decryption complete.
Extracting ATLAS_R2.1_raw.tar.gz → /kaggle/working/atlas_extracted ...


/tmp/ipykernel_16/3955974780.py:34: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(dest)


Extraction complete.


## 4. Locate the Raw Data folder
ISLES'26 requires the **Raw Data** folder specifically — not the standardised versions.

In [5]:
def find_raw_data_root(search_root: Path) -> Path:
    """
    Walk the extracted tree and return the path of the folder
    whose name contains 'Raw' (case-insensitive).
    Falls back to the root itself if no such folder is found.
    """
    for p in sorted(search_root.rglob("*")):
        if p.is_dir() and "raw" in p.name.lower():
            print(f"Found Raw Data folder: {p}")
            return p
    print(f"WARNING: No 'Raw' folder found — using extract root: {search_root}")
    return search_root


RAW_DATA_ROOT = find_raw_data_root(EXTRACT_DIR)

# Quick top-level peek
top_level = sorted(RAW_DATA_ROOT.iterdir())[:10]
print(f"\nTop-level contents ({len(top_level)} shown):")
for p in top_level:
    print(f"  {'DIR' if p.is_dir() else 'FILE'}  {p.name}")

Found Raw Data folder: /kaggle/working/atlas_extracted/Training_Raw

Top-level contents (10 shown):
  FILE  .DS_Store
  DIR  R001
  DIR  R002
  DIR  R003
  DIR  R004
  DIR  R005
  DIR  R009
  DIR  R010
  DIR  R011
  DIR  R014


## 4b. Inspect raw structure (run once to understand depth)

In [6]:
# Print a 3-level tree to understand the actual folder hierarchy
def print_tree(root: Path, max_depth: int = 3, max_items: int = 4) -> None:
    def _recurse(path, depth):
        if depth > max_depth:
            return
        children = sorted(path.iterdir())[:max_items]
        for child in children:
            indent = "  " * depth
            tag = "DIR " if child.is_dir() else "FILE"
            print(f"{indent}{tag}  {child.name}")
            if child.is_dir():
                _recurse(child, depth + 1)
        remainder = len(sorted(path.iterdir())) - max_items
        if remainder > 0:
            print(f"{'  ' * (depth)}  ... and {remainder} more")
    _recurse(root, 0)

print_tree(RAW_DATA_ROOT, max_depth=4, max_items=3)

FILE  .DS_Store
DIR   R001
  DIR   sub-r001s001
    DIR   ses-1
      FILE  .DS_Store
      DIR   anat
        FILE  sub-r001s001_ses-1_metadata.csv
        FILE  sub-r001s001_ses-1_space-orig_desc-brain_T1w.nii.gz
        FILE  sub-r001s001_ses-1_space-orig_label-lesion_desc-T1lesion_mask.nii.gz
  DIR   sub-r001s002
    DIR   ses-1
      DIR   anat
        FILE  sub-r001s002_ses-1_metadata.csv
        FILE  sub-r001s002_ses-1_space-orig_desc-brain_T1w.nii.gz
        FILE  sub-r001s002_ses-1_space-orig_label-lesion_desc-T1lesion_mask.nii.gz
  DIR   sub-r001s003
    DIR   ses-1
      DIR   anat
        FILE  sub-r001s003_ses-1_metadata.csv
        FILE  sub-r001s003_ses-1_space-orig_desc-brain_T1w.nii.gz
        FILE  sub-r001s003_ses-1_space-orig_label-lesion_desc-T1lesion_mask.nii.gz
    ... and 35 more
DIR   R002
  DIR   sub-r002s001
    DIR   ses-1
      DIR   anat
        FILE  sub-r002s001_ses-1_metadata.csv
        FILE  sub-r002s001_ses-1_space-orig_desc-brain_T1w.nii.gz
       

## 4c. Locate root-level metadata file

In [7]:
# ATLAS R2 metadata is typically a CSV/TSV at the archive root (above Training_Raw)
# Search one level above RAW_DATA_ROOT and also within it

def find_metadata_file(search_root: Path) -> Path | None:
    """Find the first CSV or TSV that looks like a participant/metadata table."""
    candidates = []
    for ext in ("*.csv", "*.tsv", "*.xlsx"):
        candidates.extend(search_root.rglob(ext))
    if not candidates:
        return None
    # Prefer files with 'participant' or 'metadata' or 'demographic' in name
    priority_keys = ["participant", "metadata", "demographic", "subject", "atlas"]
    for key in priority_keys:
        for c in candidates:
            if key in c.name.lower():
                return c
    return candidates[0]  # fallback: first found

# Search from the extract root (one level above Training_Raw)
# META_FILE = find_metadata_file(EXTRACT_DIR)

# if META_FILE:
#     print(f"Metadata file found: {META_FILE}")
#     df_meta = pd.read_csv(META_FILE, sep=None, engine="python")  # auto-detect sep
#     print(f"Shape: {df_meta.shape}")
#     print(df_meta.head())
#     print(f"Columns: {list(df_meta.columns)}")
# else:
#     print("WARNING: No metadata CSV/TSV found. Will proceed without metadata.")
#     df_meta = None

In [8]:
meta_records = []
for csv_file in sorted(RAW_DATA_ROOT.rglob("*metadata.csv")):
    df = pd.read_csv(csv_file)
    meta_records.append(df)

df_meta = pd.concat(meta_records, ignore_index=True)
assert len(df_meta) == 955, f"Expected 955 metadata rows, got {len(df_meta)}"

df_meta["CHRONICITY_DERIVED"] = df_meta["DAYS_POST_STROKE"].apply(
    lambda d: "unknown" if pd.isna(d) else
              "acute"   if d <= 7   else
              "subacute" if d <= 90 else "chronic"
)

metadata_dir = OUTPUT_DIR / "metadata"
metadata_dir.mkdir(parents=True, exist_ok=True)

df_meta.to_csv(metadata_dir / "metadata.csv", index=False)
print(f"✓ Metadata saved: {df_meta.shape}")
print(df_meta["CHRONICITY_DERIVED"].value_counts())
print(f"DAYS_POST_STROKE nulls: {df_meta['DAYS_POST_STROKE'].isna().sum()}")

✓ Metadata saved: (955, 6)
CHRONICITY_DERIVED
chronic     630
subacute    131
unknown     126
acute        68
Name: count, dtype: int64
DAYS_POST_STROKE nulls: 126


## 5. Validate and inventory the dataset

In [9]:
def inventory_dataset(raw_root: Path) -> dict:
    """
    Walk ATLAS R2 hierarchy: raw_root/<site>/<subject>/<session>/<files>
    
    For each leaf session directory, collect:
      - All T1w NIfTI files
      - All lesion/mask NIfTI files (may be multiple raters)
      - Any JSON sidecar

    Returns a list of session-level records keyed by unique ID: site_subject_session.
    """
    records = []
    stats = defaultdict(int)

    # Determine hierarchy depth by inspecting a sample path
    # Heuristic: a session dir is the deepest dir that contains .nii files directly
    def is_session_dir(path: Path) -> bool:
        return any(f.suffix in (".nii", ".gz") for f in path.iterdir() if f.is_file())

    for nii_file in sorted(raw_root.rglob("*.nii.gz")):
        session_dir = nii_file.parent
        # Build relative path parts to extract site/subject/session labels
        rel_parts = session_dir.relative_to(raw_root).parts  # e.g. ('R001', 'sub-r001s001', 'ses-1')

        site    = rel_parts[0] if len(rel_parts) >= 1 else "unknown"
        subject = rel_parts[1] if len(rel_parts) >= 2 else "unknown"
        session = rel_parts[2] if len(rel_parts) >= 3 else "ses-01"
        uid     = f"{site}__{subject}__{session}"

        # Skip if already recorded this session
        if any(r["uid"] == uid for r in records):
            continue

        # Collect all files for this session dir
        all_niis = sorted(session_dir.glob("*.nii.gz"))
        t1_files   = [f for f in all_niis if "T1w" in f.name or "t1w" in f.name.lower()]
        mask_files = [f for f in all_niis
                      if any(k in f.name.lower() for k in ("lesion", "mask", "label"))]
        json_files = sorted(session_dir.glob("*.json"))

        record = {
            "uid":        uid,
            "site":       site,
            "subject":    subject,
            "session":    session,
            "t1_files":   [str(f) for f in t1_files],
            "mask_files": [str(f) for f in mask_files],
            "json_files": [str(f) for f in json_files],
            "has_t1":     len(t1_files) == 1,       # exactly 1 expected
            "has_mask":   len(mask_files) >= 1,
            "n_raters":   len(mask_files),
            "t1_ambiguous": len(t1_files) != 1,
        }
        records.append(record)

        stats["total"] += 1
        if record["has_t1"]:   stats["ok_t1"] += 1
        if record["has_mask"]: stats["ok_mask"] += 1
        if record["t1_ambiguous"]: stats["ambiguous_t1"] += 1

    # Per-site summary
    site_counts = defaultdict(int)
    for r in records:
        site_counts[r["site"]] += 1

    return {
        "records": records,
        "stats": dict(stats),
        "site_counts": dict(sorted(site_counts.items())),
    }


inventory = inventory_dataset(RAW_DATA_ROOT)
stats = inventory["stats"]

print("=" * 50)
print(f"Total sessions    : {stats.get('total', 0)}")
print(f"With T1w (exact 1): {stats.get('ok_t1', 0)}")
print(f"With mask         : {stats.get('ok_mask', 0)}")
print(f"Ambiguous T1w     : {stats.get('ambiguous_t1', 0)}")
print("\nSessions per site:")
for site, count in inventory["site_counts"].items():
    print(f"  {site:10s}: {count}")

# Flag any sessions with unexpected rater counts
rater_counts = defaultdict(list)
for r in inventory["records"]:
    rater_counts[r["n_raters"]].append(r["uid"])
print("\nRater mask distribution:")
for n, uids in sorted(rater_counts.items()):
    print(f"  {n} mask(s): {len(uids)} sessions")
    if n not in (1, 2):  # flag unexpected counts
        print(f"    WARNING — unexpected rater count. Example: {uids[:3]}")

Total sessions    : 955
With T1w (exact 1): 955
With mask         : 955
Ambiguous T1w     : 0

Sessions per site:
  R001      : 38
  R002      : 12
  R003      : 15
  R004      : 37
  R005      : 34
  R009      : 111
  R010      : 26
  R011      : 29
  R014      : 9
  R015      : 23
  R017      : 16
  R018      : 11
  R019      : 14
  R023      : 15
  R024      : 21
  R027      : 32
  R028      : 25
  R029      : 9
  R031      : 37
  R034      : 24
  R035      : 15
  R038      : 94
  R039      : 4
  R040      : 86
  R042      : 35
  R044      : 4
  R045      : 4
  R046      : 13
  R047      : 48
  R048      : 44
  R049      : 24
  R050      : 14
  R052      : 32

Rater mask distribution:
  1 mask(s): 955 sessions


## 6. Organise into clean output structure
Copies (does not move) files into a flat, predictable layout:
```
atlas_raw/
  images/   <subject_id>_T1w.nii.gz
  masks/    <subject_id>_mask.nii.gz
  metadata/ <subject_id>.json
  inventory.json
```

In [10]:
def organise_output(inventory: dict, output_dir: Path, df_meta: pd.DataFrame | None) -> None:
    """
    Flatten the ATLAS R2 tree into:
      images/    {uid}_T1w.nii.gz
      masks/     {uid}_rater1.nii.gz  (and _rater2 if present)
      metadata/  metadata.csv         (root-level metadata, if found)
      inventory.json

    Skips sessions missing T1w or all masks.
    Keeps both rater masks — rater1/rater2 averaging is handled at training time.
    """
    images_dir   = output_dir / "images"
    masks_dir    = output_dir / "masks"
    metadata_dir = output_dir / "metadata"

    for d in [images_dir, masks_dir, metadata_dir]:
        d.mkdir(parents=True, exist_ok=True)

    copied = skipped = 0
    skip_reasons = defaultdict(list)

    for rec in inventory["records"]:
        uid = rec["uid"]

        # --- Guard: must have exactly 1 T1w and at least 1 mask ---
        if rec["t1_ambiguous"]:
            skip_reasons["ambiguous_t1"].append(uid)
            skipped += 1
            continue
        if not rec["has_t1"]:
            skip_reasons["missing_t1"].append(uid)
            skipped += 1
            continue
        if not rec["has_mask"]:
            skip_reasons["missing_mask"].append(uid)
            skipped += 1
            continue

        # --- Copy T1w ---
        t1_dst = images_dir / f"{uid}_T1w.nii.gz"
        if not t1_dst.exists():
            shutil.copy2(rec["t1_files"][0], t1_dst)

        # --- Copy masks (one per rater) ---
        for i, mask_src in enumerate(rec["mask_files"], start=1):
            mask_dst = masks_dir / f"{uid}_rater{i}.nii.gz"
            if not mask_dst.exists():
                shutil.copy2(mask_src, mask_dst)

        # --- Copy JSON sidecar if present ---
        if rec["json_files"]:
            json_dst = metadata_dir / f"{uid}.json"
            if not json_dst.exists():
                shutil.copy2(rec["json_files"][0], json_dst)

        copied += 1

    # --- Save root-level metadata CSV ---
    if df_meta is not None:
        df_meta.to_csv(metadata_dir / "metadata.csv", index=False)
        print(f"Metadata CSV saved ({len(df_meta)} rows).")
    else:
        print("No metadata CSV to save.")

    # --- Save inventory manifest ---
    manifest = {
        "total_sessions": len(inventory["records"]),
        "copied": copied,
        "skipped": skipped,
        "skip_reasons": {k: len(v) for k, v in skip_reasons.items()},
        "site_counts": inventory["site_counts"],
        "records": [
            {k: v for k, v in r.items() if k not in ("t1_files", "mask_files", "json_files")}
            for r in inventory["records"]
        ],
    }
    with open(output_dir / "inventory.json", "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"\nOrganised: {copied} sessions copied, {skipped} skipped.")
    if skip_reasons:
        for reason, uids in skip_reasons.items():
            print(f"  Skipped ({reason}): {uids}")
    print(f"Manifest saved: {output_dir / 'inventory.json'}")


organise_output(inventory, OUTPUT_DIR, df_meta)

Metadata CSV saved (955 rows).

Organised: 955 sessions copied, 0 skipped.
Manifest saved: /kaggle/working/atlas_raw/inventory.json


In [11]:
print(f"CHRONICITY nulls : {df_meta['CHRONICITY'].isna().sum()} / {len(df_meta)}")
print(f"DAYS nulls       : {df_meta['DAYS_POST_STROKE'].isna().sum()} / {len(df_meta)}")
print(f"ATLAS2_DATASET   : {df_meta['ATLAS2_DATASET'].value_counts().to_dict()}")

CHRONICITY nulls : 825 / 955
DAYS nulls       : 126 / 955
ATLAS2_DATASET   : {'Training': 655, 'Testing': 300}


In [12]:
def derive_chronicity(days: float) -> str:
    if pd.isna(days):   return "unknown"
    if days <= 7:       return "acute"
    if days <= 90:      return "subacute"
    return "chronic"

df_meta["CHRONICITY_DERIVED"] = df_meta["DAYS_POST_STROKE"].apply(derive_chronicity)
# Fix: Single quotes inside the f-string + value_counts for scannable output
print(f"CHRONICITY derived distribution:\n{df_meta['CHRONICITY_DERIVED'].value_counts(dropna=False)}")
print(f"CHRONICITY derived categories: {df_meta['CHRONICITY_DERIVED'].unique()}")

CHRONICITY derived distribution:
CHRONICITY_DERIVED
chronic     630
subacute    131
unknown     126
acute        68
Name: count, dtype: int64
CHRONICITY derived categories: ['chronic' 'unknown' 'acute' 'subacute']


## 7. Spot-check one sample

In [13]:
images_dir = OUTPUT_DIR / "images"
masks_dir  = OUTPUT_DIR / "masks"

sample_images = sorted(images_dir.glob("*.nii.gz"))[:3]
assert len(sample_images) > 0, "No images found in output dir — check earlier steps."

print("Sample output files:")
for img in sample_images:
    mask = masks_dir / img.name.replace("_T1w", "_rater1")
    img_mb  = img.stat().st_size / 1e6
    mask_mb = mask.stat().st_size / 1e6 if mask.exists() else -1
    print(f"  {img.name:50s}  {img_mb:.1f} MB   mask: {'OK' if mask.exists() else 'MISSING'}  {mask_mb:.1f} MB")

Sample output files:
  R001__sub-r001s001__ses-1_T1w.nii.gz                3.3 MB   mask: OK  0.0 MB
  R001__sub-r001s002__ses-1_T1w.nii.gz                3.4 MB   mask: OK  0.0 MB
  R001__sub-r001s003__ses-1_T1w.nii.gz                3.4 MB   mask: OK  0.0 MB


In [14]:
masks = sorted((OUTPUT_DIR / "masks").glob("*.nii.gz"))
images = sorted((OUTPUT_DIR / "images").glob("*.nii.gz"))

print(f"Mask count: {len(masks)}")
print("Sample:", [m.name for m in masks[:3]])

assert len(masks) == len(images) == 955, \
    f"Count mismatch: {len(images)} images vs {len(masks)} masks"
print(f"✓ {len(masks)} mask files confirmed.")

Mask count: 955
Sample: ['R001__sub-r001s001__ses-1_rater1.nii.gz', 'R001__sub-r001s002__ses-1_rater1.nii.gz', 'R001__sub-r001s003__ses-1_rater1.nii.gz']
✓ 955 mask files confirmed.


## 8. Disk usage summary

In [15]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/atlas_raw")

def dir_size_mb(path: Path) -> float:
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e6

# ── Re-measure all dirs fresh ─────────────────────────────────────────────────
images_dir   = OUTPUT_DIR / "images"
masks_dir    = OUTPUT_DIR / "masks"
metadata_dir = OUTPUT_DIR / "metadata"

n_images   = len(list(images_dir.glob("*.nii.gz")))
n_masks    = len(list(masks_dir.glob("*.nii.gz")))
meta_csv   = metadata_dir / "metadata.csv"

print("=== Final pre-commit audit ===")
print(f"images/    : {n_images} files   {dir_size_mb(images_dir):.0f} MB")
print(f"masks/     : {n_masks} files   {dir_size_mb(masks_dir):.0f} MB")
print(f"metadata/  : {dir_size_mb(metadata_dir):.1f} MB")

# ── Assertions — all must pass ────────────────────────────────────────────────
assert n_images == 955,       f"Expected 955 images, got {n_images}"
assert n_masks  == 955,       f"Expected 955 masks, got {n_masks}"
assert meta_csv.exists(),     "metadata.csv missing"

df = pd.read_csv(meta_csv)
assert len(df) == 955,        f"Expected 955 metadata rows, got {len(df)}"
assert "CHRONICITY_DERIVED" in df.columns, "CHRONICITY_DERIVED column missing"
assert "SESSION_ID"          in df.columns, "SESSION_ID column missing"
assert "DAYS_POST_STROKE"    in df.columns, "DAYS_POST_STROKE column missing"
assert "ATLAS2_DATASET"      in df.columns, "ATLAS2_DATASET column missing"

# ── Training-only subset check ────────────────────────────────────────────────
train_df = df[df["ATLAS2_DATASET"] == "Training"]
assert len(train_df) == 655,  f"Expected 655 training rows, got {len(train_df)}"

print(f"\nMetadata: {df.shape[0]} rows × {df.shape[1]} cols")
print(f"  Training sessions : {len(train_df)}")
print(f"  Testing sessions  : {len(df[df['ATLAS2_DATASET'] == 'Testing'])}")
print(f"  CHRONICITY_DERIVED: {df['CHRONICITY_DERIVED'].value_counts().to_dict()}")
print(f"  DAYS nulls        : {df['DAYS_POST_STROKE'].isna().sum()}")
print(f"\n✓ All assertions passed — safe to commit as Kaggle Dataset.")

print("\nReady to commit as a Kaggle Dataset.")
print("Next step: Kaggle → Datasets → New Dataset → Upload from /kaggle/working/atlas_raw")

=== Final pre-commit audit ===
images/    : 955 files   3017 MB
masks/     : 955 files   38 MB
metadata/  : 0.0 MB

Metadata: 955 rows × 6 cols
  Training sessions : 655
  Testing sessions  : 300
  CHRONICITY_DERIVED: {'chronic': 630, 'subacute': 131, 'unknown': 126, 'acute': 68}
  DAYS nulls        : 126

✓ All assertions passed — safe to commit as Kaggle Dataset.

Ready to commit as a Kaggle Dataset.
Next step: Kaggle → Datasets → New Dataset → Upload from /kaggle/working/atlas_raw
